In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.metrics.pairwise import cosine_similarity

# =====================
# 1. Load Data from DB
# =====================
db_user = "farmlinkdb3_user"
db_password = "jz3I75kAtGI60UmmBN8Li9816H9sks4v"
db_host = "dpg-d2fdscruibrs739r9o0g-a.oregon-postgres.render.com"
db_port = "5432"
db_name = "farmlinkdb3"

connection_string = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_string)

df_orders = pd.read_sql('SELECT * FROM "farmlinkApp_productorder";', engine)
df_products = pd.read_sql('SELECT * FROM "farmlinkApp_product";', engine)

# =====================
# 2. Merge Orders + Products
# =====================
df = df_orders.merge(
    df_products,
    left_on="product_id_id",
    right_on="id",
    suffixes=("_order", "_product")
)

# Only keep relevant columns
df = df[["farmer_id_id_order", "product_id_id", "product_name", "product_image"]]

# =====================
# 3. Create User-Item Matrix
# =====================
# One row per user, one column per product, values = purchase count
user_item_matrix = df.groupby(["farmer_id_id_order", "product_id_id"]).size().unstack(fill_value=0)

# =====================
# 4. Compute Similarities Between Users
# =====================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

# =====================
# 5. Recommend for a Given Farmer
# =====================
def recommend_products(user_id, top_n=5):
    if user_id not in user_similarity_df.index:
        return []
    
    # Find similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).index[1:]
    
    # Products bought by similar users but not by current user
    user_purchases = set(df[df["farmer_id_id_order"] == user_id]["product_id_id"])
    recommendations = []
    
    for sim_user in similar_users:
        sim_user_products = df[df["farmer_id_id_order"] == sim_user][["product_id_id", "product_name", "product_image"]]
        for _, row in sim_user_products.iterrows():
            if row["product_id_id"] not in user_purchases:
                recommendations.append(row)
        if len(recommendations) >= top_n:
            break
    
    # Remove duplicates
    recommendations_df = pd.DataFrame(recommendations).drop_duplicates(subset="product_id_id")
    return recommendations_df.head(top_n)

# =====================
# 6. Example Usage
# =====================
user_id_to_recommend = 1  # replace with real farmer ID
recommended_products = recommend_products(user_id_to_recommend, top_n=5)

print(recommended_products)


   product_id_id product_name  \
1              2        Beans   

                                       product_image  
1  https://res.cloudinary.com/dc68huvjj/image/upl...  
